# Week 6, Day 5: Fine-tune and call a memorabilia pricer

**Lesson:** [week6/day5.ipynb](../../../week6/day5.ipynb), *Fine-tuning a frontier model*. **About 25 minutes.**

**What you practice here**

- Training examples as short chats, the question from the user and the price from the assistant, written as a JSONL file of one `{"messages": [...]}` object per line.
- Uploading the files for fine-tuning with `files.create`, and starting the job with `fine_tuning.jobs.create` and the lesson's settings.
- Finding the fine-tuned model's name with `fine_tuning.jobs.retrieve`, and calling that model with the training prompt and a small token limit.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week6/day5", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson fine-tunes `gpt-4.1-nano` to price products. Each training example is a short chat in which the user asks
for the price and the assistant gives it. The chats go into JSONL files, the files are uploaded to OpenAI, and a
fine-tuning job runs on them. Once the job succeeds, the lesson asks the new model the same question without the
answer. Build those steps for wrestling memorabilia listings, each with a `summary` in Day 2's format and a `price`.

The functions that call the API, `start_fine_tune` and `fine_tuned_pricer`, take an `OpenAI()` client as their first
parameter and make the calls themselves. The checks pass a stand-in shaped like it: it records the files and the job
you send and answers with canned replies, so nothing is uploaded and nothing is spent.

## What each call should give

```python
messages_for(belt)
# [{'role': 'user', 'content': 'Estimate the price of this item...\n\nTitle: Replica Winged Eagle championship belt\n...'},
#  {'role': 'assistant', 'content': '$399.99'}]

print(make_jsonl([program, belt]))
# {"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "$85.00"}]}
# {"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "$399.99"}]}

job_id = start_fine_tune(client, "train.jsonl", "validation.jsonl")
# uploads both files, starts the job on them, and returns its id: 'ftjob-abc123'

pricer = fine_tuned_pricer(client, job_id)
pricer(belt)
# asks the fine-tuned model the training question, and returns its reply: '$410.00'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import json

### 1 · `messages_for(listing)`

Turns a listing into one training example: the question from the user and the price from the assistant, as the
lesson's `messages_for` does. Return two messages and no system message. First a user message with your instruction
asking for the price, then a blank line, then `listing.summary`. Then an assistant message with the price as the
lesson writes it: a dollar sign and exactly two decimals, with no comma, so 12 gives `"$12.00"`, 1234.5 gives
`"$1234.50"` and 9.999 gives `"$10.00"`.

<details><summary>Hint</summary>

The lesson formats the price with `f"${item.price:.2f}"`.

```python
[{"role": "user", "content": instruction + "\n\n" + listing.summary},
 {"role": "assistant", "content": price_text}]
```

</details>

In [ ]:
def messages_for(listing):
    # Write messages_for here, as described above.
    raise NotImplementedError("messages_for is not written yet")

In [ ]:
check("messages_for")

### 2 · `make_jsonl(listings)`

Writes the text of a training file, one chat per line, as the lesson's `make_jsonl` does. For each listing, in
order, make one line holding a JSON object whose only key, `"messages"`, holds `messages_for(listing)`, written with
the `json` module so that the new lines inside a summary are escaped. Join the lines with new lines and leave none at
the end. No listings give an empty text.

<details><summary>Hint</summary>

```python
json.dumps({"messages": messages_for(listing)})
"\n".join(lines)
```

</details>

In [ ]:
def make_jsonl(listings):
    # Write make_jsonl here, as described above.
    raise NotImplementedError("make_jsonl is not written yet")

In [ ]:
check("make_jsonl")

### 3 · `start_fine_tune(client, train_path, validation_path, model="gpt-4.1-nano-2025-04-14")`

Uploads the training and validation files and starts a fine-tuning job on them, as the lesson's Steps 1 and 2 do,
and returns the job's id. Open each file in binary mode and upload it with the client's file upload, for the purpose
`"fine-tune"`. Then create a fine-tuning job with the two uploads' ids as its training file and its validation file,
the model `model`, and the lesson's settings: a `seed` of 42, the `hyperparameters` `n_epochs` 1 and `batch_size`
1, and the `suffix` `"pricer"`. Return the id of the job it gives back.

<details><summary>Hint</summary>

```python
with open(train_path, "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
job = client.fine_tuning.jobs.create(training_file=train_file.id, validation_file=validation_file.id, model=model,
                                     seed=42, hyperparameters={"n_epochs": 1, "batch_size": 1}, suffix="pricer")
return job.id
```

</details>

In [ ]:
def start_fine_tune(client, train_path, validation_path, model="gpt-4.1-nano-2025-04-14"):
    # Write start_fine_tune here, as described above.
    raise NotImplementedError("start_fine_tune is not written yet")

In [ ]:
check("start_fine_tune")

### 4 · `fine_tuned_pricer(client, job_id)`

Looks up the fine-tuned model of a finished job and returns a pricer that asks it, as the lesson's Step 3 does.
Retrieve the job by its id in `fine_tuned_pricer` itself, not each time the pricer runs, and read the name of its
fine-tuned model. It stays `None` until the job's status is `"succeeded"`. While it is `None`, raise `ValueError`
with a message that names the job's status. Otherwise return a function that takes a listing, makes one Chat
Completions call with the fine-tuned model, only the user message of `messages_for(listing)` and a limit of 7 tokens
for the reply, as the lesson does, and returns the reply's text. The model comes from the job, so this part takes no
`model`.

<details><summary>Hint</summary>

```python
job = client.fine_tuning.jobs.retrieve(job_id)
job.fine_tuned_model, job.status
response = client.chat.completions.create(model=name, messages=messages_for(listing)[:1], max_tokens=7)
return response.choices[0].message.content
```

</details>

In [ ]:
def fine_tuned_pricer(client, job_id):
    # Write fine_tuned_pricer here, as described above.
    raise NotImplementedError("fine_tuned_pricer is not written yet")

In [ ]:
check("fine_tuned_pricer")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 6, Day 5 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Fine-tuning `gpt-4.1-nano` on 100 examples for one epoch costs a few cents, and the job takes from a few minutes to an hour, depending on OpenAI's queue.

1. Load your keys with `load_dotenv(override=True)` and make `client = OpenAI()`.
2. Put the course's `week6` folder on the path with `sys.path.insert(0, str(ROOT / "week6"))`, load the lesson's
   data with `Item.from_hub("ed-donner/items_lite")`, and write `make_jsonl(train[:100])` and
   `make_jsonl(val[:50])` to two files in a temporary folder.
3. Start the job with `start_fine_tune(client, ...)`. Until its `status` is `succeeded`, `failed` or `cancelled`,
   print its latest events every minute with `client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id,
   limit=3)`, as the lesson's monitoring cells do.
4. Make `pricer = fine_tuned_pricer(client, job_id)`, print `pricer(test[0])` next to `test[0].price`, and score it
   with the lesson's `evaluate(pricer, test)`.

<details><summary>Stuck? One way to write it</summary>

```python
import sys
import tempfile
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()

sys.path.insert(0, str(ROOT / "week6"))  # the lesson's pricer package
from pricer.evaluator import evaluate
from pricer.items import Item

train, val, test = Item.from_hub("ed-donner/items_lite")
folder = Path(tempfile.mkdtemp())
for name, chosen in (("fine_tune_train.jsonl", train[:100]), ("fine_tune_validation.jsonl", val[:50])):
    with open(folder / name, "w", encoding="utf-8") as f:
        f.write(make_jsonl(chosen))

job_id = start_fine_tune(client, folder / "fine_tune_train.jsonl", folder / "fine_tune_validation.jsonl")
print("Started", job_id)
while client.fine_tuning.jobs.retrieve(job_id).status not in ("succeeded", "failed", "cancelled"):
    for event in client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=3).data:
        print(event.message)
    time.sleep(60)

pricer = fine_tuned_pricer(client, job_id)
print(pricer(test[0]), "for an item that costs", test[0].price)
evaluate(pricer, test)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.